# Phase 2: Feature Extraction for Animal Classification

This notebook implements feature extraction for animal classification using SVM and Decision Tree models. The features are optimized based on Phase 1 analysis to balance accuracy and computational efficiency.

**Total Features**: ~730 (optimized from ~1455)

**Features Extracted**:
- LBP (Local Binary Pattern) - Texture analysis (~18 features)
- GLCM (Haralick Features) - Global texture (4 features)
- Color Histogram (HSV) - Color analysis (48 features)
- HOG (Histogram of Oriented Gradients) - Shape analysis (~648 features)
- Statistical Features - Brightness, contrast, etc. (7 features)
- Shape Features - Aspect ratio, compactness (3 features)
- Edge Features - Edge density and variance (2 features)

In [1]:
# Import Required Libraries
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops, hog
from scipy.stats import skew, kurtosis
from scipy import ndimage

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ==========================
# Global Configuration
# ==========================

DATA_DIR = r"Dataset\Animals-10"
OUTPUT_CSV = r"CSV\features.csv"

CLASSES = [
    "butterfly", "cat", "chicken", "cow", "dog",
    "elephant", "horse", "sheep", "spider", "squirrel"
]

IMG_SIZE = (128, 128)  # Image size for standardization

# LBP Parameters (for texture detection) - Optimized
LBP_RADIUS = 2  # Reduced from 3 to 2
LBP_POINTS = 8 * LBP_RADIUS  # 16 points
LBP_METHOD = "uniform"

# Color Histogram (for animal color differentiation) - Optimized
COLOR_BINS = 16  # Reduced from 32 to 16 (sufficient for color distinction)

# GLCM - Haralick Features (for texture) - Optimized
GLCM_DIST = [1]  # Only one distance (most important)
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]  # 4 angles

# HOG Parameters (for shape and structure) - Optimized
HOG_ORIENTATIONS = 8  # Reduced from 9 to 8
HOG_PIXELS_PER_CELL = (16, 16)
HOG_CELLS_PER_BLOCK = (2, 2)

print("Configuration loaded:")
print(f"- Dataset: {DATA_DIR}")
print(f"- Classes: {len(CLASSES)}")
print(f"- Image size: {IMG_SIZE}")
print(f"- Total features: ~730 (optimized)")

Configuration loaded:
- Dataset: Dataset\Animals-10
- Classes: 10
- Image size: (128, 128)
- Total features: ~730 (optimized)


## Feature Extraction Functions

This section contains all the feature extraction functions optimized for animal classification.

In [3]:
# ==========================
# Feature Extraction Functions
# ==========================

# 1. LBP - Local Binary Pattern (for texture detection)
def extract_lbp(gray):
    """
    Extract Local Binary Pattern features
    Suitable for: Texture detection (skin, fur, feathers)
    """
    lbp = local_binary_pattern(gray, LBP_POINTS, LBP_RADIUS, method=LBP_METHOD)
    n_bins = LBP_POINTS + 2
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins+1), density=True)
    return hist.astype(np.float32)


# 2. GLCM - Gray Level Co-occurrence Matrix (for texture and patterns)
def extract_glcm(gray):
    """
    Extract Haralick features from GLCM - Only most important ones
    Suitable for: Texture, roughness, uniformity
    """
    gray_u8 = gray.astype(np.uint8)
    glcm = graycomatrix(gray_u8, distances=GLCM_DIST, angles=GLCM_ANGLES,
                        levels=256, symmetric=True, normed=True)

    features = []
    # Only most important GLCM features (removed ASM and dissimilarity with low impact)
    props = ["contrast", "homogeneity", "energy", "correlation"]

    for p in props:
        feat = graycoprops(glcm, p)
        features.append(feat.mean())  # Only mean (removed std)

    return np.array(features, dtype=np.float32)


# 3. Color Histogram (for animal color differentiation)
def extract_color_hist(bgr):
    """
    Extract color histogram in HSV space
    Suitable for: Color differentiation between animals (brown, black, white, colorful)
    """
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    hist_feats = []

    ranges = [(0, 180), (0, 256), (0, 256)]

    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [COLOR_BINS], ranges[ch])
        h = cv2.normalize(h, h).flatten()
        hist_feats.append(h)

    return np.concatenate(hist_feats).astype(np.float32)


# 4. HOG - Histogram of Oriented Gradients (for shape and structure)
def extract_hog_features(gray):
    """
    Extract HOG features
    Suitable for: Body shape detection, ears, tail, wings
    """
    hog_feat = hog(gray,
                   orientations=HOG_ORIENTATIONS,
                   pixels_per_cell=HOG_PIXELS_PER_CELL,
                   cells_per_block=HOG_CELLS_PER_BLOCK,
                   block_norm='L2-Hys',
                   visualize=False,
                   feature_vector=True)
    return hog_feat.astype(np.float32)


# 5. Statistical Features (based on Phase 1 findings)
def extract_statistical_features(gray, bgr):
    """
    Extract important statistical features from image
    Based on Phase 1 results: brightness, contrast, variance
    """
    features = []

    # Brightness & Contrast (very important for animal differentiation)
    brightness = np.mean(gray)
    contrast = np.std(gray)
    features.extend([brightness, contrast])

    # RGB Channel Means only (removed stds)
    for ch in range(3):
        channel = bgr[:, :, ch]
        features.append(np.mean(channel))

    # Variance (complexity)
    variance = np.var(gray)
    features.append(variance)

    # Saturation (important for butterfly vs others distinction)
    max_rgb = np.max(bgr, axis=2)
    min_rgb = np.min(bgr, axis=2)
    saturation = np.mean((max_rgb - min_rgb) / (max_rgb + 1e-7))
    features.append(saturation)

    return np.array(features, dtype=np.float32)


# 6. Shape Features (simplified)
def extract_shape_features(gray):
    """
    Extract main shape features
    Suitable for: Different animal shape differentiation
    """
    features = []

    # Edge detection
    edges = cv2.Canny(gray, 50, 150)

    # Edge pixel ratio (important)
    edge_ratio = np.sum(edges > 0) / edges.size
    features.append(edge_ratio)

    # Contour analysis
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest_contour = max(contours, key=cv2.contourArea)

        # Aspect ratio (important for shape distinction)
        x, y, w, h = cv2.boundingRect(largest_contour)
        aspect_ratio = w / h if h > 0 else 0
        features.append(aspect_ratio)

        # Compactness (important)
        area = cv2.contourArea(largest_contour)
        perimeter = cv2.arcLength(largest_contour, True)
        if perimeter > 0:
            compactness = (4 * np.pi * area) / (perimeter ** 2)
        else:
            compactness = 0
        features.append(compactness)
    else:
        features.extend([0, 0])

    return np.array(features, dtype=np.float32)


# 7. Edge Features (simplified)
def extract_edge_features(gray):
    """
    Extract edge features - Only most important ones
    """
    features = []

    # Sobel edges
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    # Gradient magnitude
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

    # Only mean and std (most important)
    features.append(np.mean(magnitude))
    features.append(np.std(magnitude))

    return np.array(features, dtype=np.float32)

print("Feature extraction functions defined successfully!")

Feature extraction functions defined successfully!


## Image Loading and Preprocessing

In [4]:
# ==========================
# Image Loading and Preprocessing
# ==========================

def load_image(img_path):
    """Load and preprocess image"""
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read image {img_path}")

    # Convert grayscale images to BGR
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # Remove alpha channel
    if img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

    # Resize to standard size
    img = cv2.resize(img, IMG_SIZE)
    return img

print("Image loading function defined!")

Image loading function defined!


## Main Feature Extraction Function

In [5]:
# ==========================
# Main Feature Extraction Function
# ==========================

def extract_all_features(img_path):
    """
    Extract all optimized features from an image

    Features extracted (optimized for SVM and Decision Tree):
    1. LBP: Local texture (~18 features)
    2. GLCM: Haralick features (4 features - most important only)
    3. Color Histogram: HSV color histogram (48 features)
    4. HOG: Shape and structure (~648 features)
    5. Statistical: Statistical features (7 features)
    6. Shape: Shape features (3 features)
    7. Edge: Edge features (2 features)

    Total: ~730 features (50% reduction from previous version)
    """
    img = load_image(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Extract all features
    lbp = extract_lbp(gray)
    glcm = extract_glcm(gray)
    color = extract_color_hist(img)
    hog_feat = extract_hog_features(gray)
    stats = extract_statistical_features(gray, img)
    shape = extract_shape_features(gray)
    edge = extract_edge_features(gray)

    # Combine all features
    all_features = np.concatenate([
        lbp,      # Local texture (important for skin/fur/feather distinction)
        glcm,     # Global texture (important for patterns)
        color,    # Color (very important - from Phase 1)
        hog_feat, # Shape (important for animal distinction)
        stats,    # Statistical (important - brightness, contrast, saturation)
        shape,    # Shape (aspect ratio, compactness)
        edge      # Edge (edge density)
    ])

    return all_features

print("Main feature extraction function defined!")

Main feature extraction function defined!


## CSV Building Function

In [6]:
# ==========================
# Build CSV with All Features
# ==========================

def build_features_csv(data_dir, out_csv="features.csv"):
    """
    Extract features from all images and save to CSV
    """
    print("=" * 70)
    print("Starting Feature Extraction - Phase 2")
    print("=" * 70)

    all_data = []
    feature_names = None
    total_processed = 0
    total_errors = 0

    for label_idx, cls in enumerate(CLASSES):
        class_dir = os.path.join(data_dir, cls)
        print(f"\n[INFO] Processing class: {cls} (label {label_idx})")

        if not os.path.exists(class_dir):
            print(f"[WARNING] Directory {class_dir} does not exist!")
            continue

        image_files = [f for f in os.listdir(class_dir)
                      if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]

        print(f"  Number of images: {len(image_files)}")

        for idx, fname in enumerate(image_files):
            path = os.path.join(class_dir, fname)

            try:
                # Extract features
                feat = extract_all_features(path)

                # Create feature names on first run
                if feature_names is None:
                    n_features = len(feat)
                    feature_names = ["path", "class", "label"] + [f"feature_{i}" for i in range(n_features)]
                    print(f"  [INFO] Total features: {n_features}")

                # Save data
                row = [path, cls, label_idx] + feat.tolist()
                all_data.append(row)
                total_processed += 1

                # Progress
                if (idx + 1) % 100 == 0:
                    print(f"    Processed: {idx + 1}/{len(image_files)}")

            except Exception as e:
                print(f"  [ERROR] Error in {fname}: {e}")
                total_errors += 1
                continue

    # Save to DataFrame and CSV
    print("\n" + "=" * 70)
    print("Saving results to CSV...")
    df = pd.DataFrame(all_data, columns=feature_names)
    df.to_csv(out_csv, index=False)

    print(f"[SUCCESS] File saved: {out_csv}")
    print(f"  Total images processed: {total_processed}")
    print(f"  Total errors: {total_errors}")
    print(f"  Number of features: {len(feature_names) - 3}")  # -3 for path, class, label
    print(f"  DataFrame shape: {df.shape}")
    print("=" * 70)

    # Show class distribution
    print("\nClass distribution:")
    print(df['class'].value_counts().sort_index())

    # Show sample data
    print("\nSample data:")
    print(df[['class', 'label']].head(10))

    return df

print("CSV building function defined!")

CSV building function defined!


## Execute Feature Extraction

In [7]:
# ==========================
# Execute Feature Extraction
# ==========================

print("=" * 70)
print("Phase 2: Optimized Feature Extraction for Animal Classification")
print("=" * 70)
print("\nExtracted Features (optimized for SVM and Decision Tree):")
print("  ✓ LBP (Local Binary Pattern) - Local texture (~18 features)")
print("  ✓ GLCM (Haralick) - Global texture (4 features)")
print("  ✓ Color Histogram (HSV) - Color (48 features)")
print("  ✓ HOG - Shape and structure (~648 features)")
print("  ✓ Statistical - From Phase 1 (7 features)")
print("  ✓ Shape - Shape features (3 features)")
print("  ✓ Edge - Edge features (2 features)")
print("\n  📊 Total: ~730 features (optimized)")
print("=" * 70)
print("\n")

# Execute feature extraction
df = build_features_csv(DATA_DIR, out_csv=OUTPUT_CSV)

print("\n" + "=" * 70)
print("✅ Phase 2 completed successfully!")
print(f"📊 Output file: {OUTPUT_CSV}")
print(f"📈 Number of samples: {len(df)}")
print(f"🎯 Number of features: {df.shape[1] - 3}")
print("=" * 70)

Phase 2: Optimized Feature Extraction for Animal Classification

Extracted Features (optimized for SVM and Decision Tree):
  ✓ LBP (Local Binary Pattern) - Local texture (~18 features)
  ✓ GLCM (Haralick) - Global texture (4 features)
  ✓ Color Histogram (HSV) - Color (48 features)
  ✓ HOG - Shape and structure (~648 features)
  ✓ Statistical - From Phase 1 (7 features)
  ✓ Shape - Shape features (3 features)
  ✓ Edge - Edge features (2 features)

  📊 Total: ~730 features (optimized)


Starting Feature Extraction - Phase 2

[INFO] Processing class: butterfly (label 0)
  Number of images: 2112
  [INFO] Total features: 1650
    Processed: 100/2112
    Processed: 100/2112
    Processed: 200/2112
    Processed: 200/2112
    Processed: 300/2112
    Processed: 300/2112
    Processed: 400/2112
    Processed: 400/2112
    Processed: 500/2112
    Processed: 500/2112
    Processed: 600/2112
    Processed: 600/2112
    Processed: 700/2112
    Processed: 700/2112
    Processed: 800/2112
    Proces